In [1]:
from transformers import pipeline
import pandas as pd
import os
from huggingface_hub import InferenceClient


C:\Users\geraa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [24]:
from dotenv import load_dotenv

In [2]:
import torch

In [3]:
from transformers import AutoTokenizer

In [4]:
from transformers import AutoModel

In [29]:
load_dotenv()  
token = os.getenv("HF_TOKEN")

In [27]:
model_name = "PKOBP/polish-roberta-8k"

In [30]:
tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)

In [31]:
model = AutoModel.from_pretrained(model_name, token=token)

C:\Users\geraa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\geraa\.cache\huggingface\hub\models--PKOBP--polish-roberta-8k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██

In [34]:
df = pd.read_json('book_engagement_metrics.json', lines=True)
print(df.columns)

Index(['id', 'like_count', 'region_code', 'video_description',
       'favorites_count', 'hashtag_names', 'effect_ids', 'is_stem_verified',
       'playlist_id', 'username', 'video_duration', 'video_label',
       'comment_count', 'create_time', 'music_id', 'view_count',
       'video_mention_list', 'voice_to_text', 'hashtag_info_list',
       'share_count', 'sticker_info_list', 'effect_info_list', 'video_tag',
       'full_text', 'book_title', 'view_count_pct_rank', 'like_count_pct_rank',
       'share_count_pct_rank', 'favorites_count_pct_rank',
       'comment_count_pct_rank', 'raw_engagement_metrics',
       'engagement_metrics'],
      dtype='object')


In [43]:
# Preparing the text for the model
df['text_for_sentiment'] = (
    df['full_text']
    .fillna(df['video_description'])
    .fillna(df['voice_to_text'])
    .fillna('')
)
df_clean_text = df[df['text_for_sentiment'].str.strip() != ''].copy()

In [44]:
texts = df_clean_text['text_for_sentiment'].tolist()

In [ ]:
batch_size = 8
all_embeddings = []

In [54]:
for i in range(0, len(texts), batch_size):
    batch_texts = texts[i : i + batch_size]
    
    inputs = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=8192,
        return_tensors="pt"
    )
    
    with torch.no_grad():
        outputs = model(**inputs)
        embeddings = outputs.last_hidden_state.mean(dim=1)
        all_embeddings.append(embeddings)

KeyboardInterrupt: 

In [ ]:

print("Shape:", embeddings.shape)
